# 🛒 Zepto & Blinkit Chennai — Customer Segmentation ML Project
### Using K-Means Clustering | Google Colab Notebook

---
**Goal:** Segment Chennai customers of Zepto & Blinkit into meaningful groups based on their ordering behavior, so the business can personalize marketing, offers, and retention strategies.

**Pipeline:**
1. Generate realistic synthetic Chennai dataset
2. Exploratory Data Analysis (EDA)
3. Feature Engineering & Preprocessing
4. Optimal K selection (Elbow + Silhouette)
5. K-Means Clustering
6. Cluster Profiling & Insights
7. Save model artifacts for Streamlit app


## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install required libraries
!pip install scikit-learn pandas numpy matplotlib seaborn plotly joblib -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import joblib

np.random.seed(42)
print('✅ All libraries imported successfully!')

## 🏙️ Step 2: Generate Realistic Chennai Dataset

In [ ]:
def generate_chennai_data(n=2000):
    np.random.seed(42)

    # Chennai localities with Zepto/Blinkit presence
    localities = [
        'Anna Nagar', 'T Nagar', 'Adyar', 'Velachery', 'OMR',
        'Porur', 'Chromepet', 'Tambaram', 'Nungambakkam', 'Mylapore',
        'Sholinganallur', 'Perungudi', 'Guindy', 'Kodambakkam', 'Ambattur'
    ]

    platforms = ['Zepto', 'Blinkit', 'Both']
    preferred_categories = [
        'Groceries', 'Snacks & Beverages', 'Dairy & Eggs',
        'Fruits & Vegetables', 'Personal Care', 'Household Items'
    ]
    payment_modes = ['UPI', 'Credit Card', 'Debit Card', 'Cash on Delivery', 'Wallet']
    time_slots = ['Morning (6-10am)', 'Afternoon (12-4pm)', 'Evening (4-8pm)', 'Night (8-11pm)']
    genders = ['Male', 'Female']
    age_groups = ['18-24', '25-34', '35-44', '45-54', '55+']

    data = {
        'customer_id': [f'CUST{str(i).zfill(4)}' for i in range(1, n + 1)],
        'age': np.random.choice([22, 28, 35, 45, 58], size=n, p=[0.25, 0.35, 0.20, 0.12, 0.08]),
        'gender': np.random.choice(genders, size=n, p=[0.52, 0.48]),
        'locality': np.random.choice(localities, size=n),
        'platform': np.random.choice(platforms, size=n, p=[0.38, 0.42, 0.20]),

        # Behavioral features
        'monthly_orders': np.clip(np.random.normal(14, 6, n).astype(int), 1, 45),
        'avg_order_value': np.clip(np.random.normal(420, 180, n), 80, 1500).round(2),
        'monthly_spend': None,  # derived below
        'avg_delivery_time_min': np.clip(np.random.normal(18, 5, n), 8, 45).round(1),
        'num_categories_ordered': np.random.randint(1, 7, n),
        'reorder_rate': np.clip(np.random.beta(3, 2, n), 0.1, 1.0).round(2),
        'discount_usage_pct': np.clip(np.random.beta(2, 3, n) * 100, 0, 100).round(1),
        'app_sessions_per_week': np.clip(np.random.normal(8, 4, n), 1, 30).round(1),
        'ratings_given': np.random.randint(0, 30, n),
        'complaints_filed': np.random.randint(0, 6, n),
        'membership': np.random.choice([0, 1], size=n, p=[0.55, 0.45]),

        # Categorical
        'preferred_category': np.random.choice(preferred_categories, size=n),
        'preferred_payment': np.random.choice(payment_modes, size=n, p=[0.45, 0.22, 0.15, 0.10, 0.08]),
        'preferred_time_slot': np.random.choice(time_slots, size=n),
        'days_since_last_order': np.clip(np.random.exponential(5, n).astype(int), 0, 60),
        'tenure_months': np.random.randint(1, 36, n),
    }

    df = pd.DataFrame(data)
    df['monthly_spend'] = (df['monthly_orders'] * df['avg_order_value']).round(2)

    # Inject realistic correlations: high spenders order more often, use membership
    high_spend_mask = df['avg_order_value'] > 600
    df.loc[high_spend_mask, 'membership'] = np.random.choice([0, 1], size=high_spend_mask.sum(), p=[0.2, 0.8])
    df.loc[high_spend_mask, 'monthly_orders'] = np.clip(
        df.loc[high_spend_mask, 'monthly_orders'] + np.random.randint(3, 8, high_spend_mask.sum()), 1, 45
    )

    return df

df = generate_chennai_data(2000)
print(f'✅ Dataset created: {df.shape[0]} customers, {df.shape[1]} features')
df.head()

## 📊 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
print(df.info())
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Numerical Summary ===')
df.describe().round(2)

In [ ]:
# Platform distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Chennai Quick Commerce — Customer EDA', fontsize=16, fontweight='bold')

# Platform split
colors = ['#00B4D8', '#FF6B35', '#8338EC']
df['platform'].value_counts().plot(kind='pie', ax=axes[0,0], colors=colors,
    autopct='%1.1f%%', startangle=90)
axes[0,0].set_title('Platform Distribution')

# Monthly orders distribution
axes[0,1].hist(df['monthly_orders'], bins=30, color='#00B4D8', edgecolor='white')
axes[0,1].set_title('Monthly Orders Distribution')
axes[0,1].set_xlabel('Orders per Month')

# Avg order value
axes[0,2].hist(df['avg_order_value'], bins=30, color='#FF6B35', edgecolor='white')
axes[0,2].set_title('Avg Order Value (₹) Distribution')
axes[0,2].set_xlabel('₹ per Order')

# Monthly spend by platform
df.groupby('platform')['monthly_spend'].mean().plot(kind='bar', ax=axes[1,0],
    color=colors, edgecolor='white')
axes[1,0].set_title('Avg Monthly Spend by Platform')
axes[1,0].set_ylabel('₹')
axes[1,0].tick_params(axis='x', rotation=0)

# Top localities
df['locality'].value_counts().head(8).plot(kind='barh', ax=axes[1,1], color='#8338EC')
axes[1,1].set_title('Top 8 Localities by Customer Count')

# Payment mode
df['preferred_payment'].value_counts().plot(kind='bar', ax=axes[1,2],
    color=['#06D6A0', '#FFD166', '#EF476F', '#118AB2', '#073B4C'])
axes[1,2].set_title('Preferred Payment Mode')
axes[1,2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA charts saved')

In [ ]:
# Correlation heatmap
num_cols = ['monthly_orders', 'avg_order_value', 'monthly_spend',
            'reorder_rate', 'discount_usage_pct', 'app_sessions_per_week',
            'num_categories_ordered', 'membership', 'tenure_months',
            'days_since_last_order', 'complaints_filed']

plt.figure(figsize=(12, 8))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## ⚙️ Step 4: Feature Engineering & Preprocessing

In [ ]:
# RFM-style features for clustering
features = [
    'monthly_orders',
    'avg_order_value',
    'monthly_spend',
    'reorder_rate',
    'discount_usage_pct',
    'app_sessions_per_week',
    'num_categories_ordered',
    'membership',
    'tenure_months',
    'days_since_last_order',
    'complaints_filed',
    'ratings_given'
]

X = df[features].copy()

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'✅ Feature matrix shape: {X_scaled.shape}')
print(f'Features used: {features}')

## 🔍 Step 5: Find Optimal K (Elbow + Silhouette)

In [ ]:
inertias = []
silhouette_scores = []
K_range = range(2, 11)

print('Finding optimal K...')
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouette_scores.append(sil)
    print(f'  K={k} | Inertia: {km.inertia_:.0f} | Silhouette: {sil:.4f}')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Optimal K Selection', fontsize=14, fontweight='bold')

ax1.plot(list(K_range), inertias, 'bo-', linewidth=2, markersize=8)
ax1.axvline(x=4, color='red', linestyle='--', label='Chosen K=4')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.set_title('Elbow Method')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(list(K_range), silhouette_scores, 'rs-', linewidth=2, markersize=8)
ax2.axvline(x=4, color='blue', linestyle='--', label='Chosen K=4')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('optimal_k.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Best Silhouette Score: {max(silhouette_scores):.4f} at K={silhouette_scores.index(max(silhouette_scores))+2}')

## 🤖 Step 6: Train K-Means with K=4

In [ ]:
OPTIMAL_K = 4

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=15, max_iter=500)
df['cluster'] = kmeans.fit_predict(X_scaled)

final_silhouette = silhouette_score(X_scaled, df['cluster'])
print(f'✅ K-Means trained with K={OPTIMAL_K}')
print(f'   Final Silhouette Score: {final_silhouette:.4f}')
print(f'   Inertia: {kmeans.inertia_:.2f}')
print(f'\nCluster Distribution:')
print(df['cluster'].value_counts().sort_index())

## 📈 Step 7: Visualize Clusters (PCA 2D)

In [ ]:
# PCA for 2D visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df['pca1'] = X_pca[:, 0]
df['pca2'] = X_pca[:, 1]

print(f'Variance explained by 2 PCs: {pca.explained_variance_ratio_.sum()*100:.1f}%')

# Plot
cluster_colors = ['#FF6B35', '#00B4D8', '#8338EC', '#06D6A0']
cluster_labels_temp = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3']

plt.figure(figsize=(12, 8))
for c in range(OPTIMAL_K):
    mask = df['cluster'] == c
    plt.scatter(df.loc[mask, 'pca1'], df.loc[mask, 'pca2'],
                c=cluster_colors[c], label=cluster_labels_temp[c],
                alpha=0.6, s=30)

# Plot centroids
centers_pca = pca.transform(kmeans.cluster_centers_)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1],
            c='black', marker='X', s=300, zorder=5, label='Centroids')

plt.title('Customer Segments — PCA 2D Projection\n(Zepto & Blinkit Chennai)', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cluster_pca.png', dpi=150, bbox_inches='tight')
plt.show()

## 🏷️ Step 8: Cluster Profiling & Naming

In [ ]:
# Profile each cluster
profile_cols = ['monthly_orders', 'avg_order_value', 'monthly_spend',
                'reorder_rate', 'discount_usage_pct', 'app_sessions_per_week',
                'membership', 'tenure_months', 'days_since_last_order']

cluster_profile = df.groupby('cluster')[profile_cols].mean().round(2)
print('=== Cluster Profiles ===')
print(cluster_profile.to_string())

In [ ]:
# Assign meaningful names based on profiling
# You may re-order these based on your actual cluster output
cluster_names = {
    0: '💎 Premium Loyalists',
    1: '🧪 Occasional Browsers',
    2: '🔥 Bargain Hunters',
    3: '⭐ Regular Mainstream'
}

df['segment'] = df['cluster'].map(cluster_names)

# Summary table
summary = df.groupby('segment').agg(
    customers=('customer_id', 'count'),
    avg_monthly_orders=('monthly_orders', 'mean'),
    avg_order_value=('avg_order_value', 'mean'),
    avg_monthly_spend=('monthly_spend', 'mean'),
    reorder_rate=('reorder_rate', 'mean'),
    discount_usage=('discount_usage_pct', 'mean'),
    membership_pct=('membership', 'mean'),
).round(2)

print('\n=== Customer Segment Summary ===')
print(summary.to_string())

In [ ]:
# Radar chart for segment comparison
radar_features = ['monthly_orders', 'avg_order_value', 'reorder_rate',
                  'discount_usage_pct', 'app_sessions_per_week', 'membership']

# Normalize 0-1 per feature
radar_data = df.groupby('segment')[radar_features].mean()
radar_norm = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min())

angles = np.linspace(0, 2 * np.pi, len(radar_features), endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(1, 4, figsize=(22, 6), subplot_kw=dict(polar=True))
fig.suptitle('Segment Radar Profiles — Zepto & Blinkit Chennai', fontsize=14, fontweight='bold')

colors = ['#FF6B35', '#00B4D8', '#8338EC', '#06D6A0']
for idx, (seg, row) in enumerate(radar_norm.iterrows()):
    values = row.tolist() + row.tolist()[:1]
    ax = axes[idx]
    ax.plot(angles, values, color=colors[idx], linewidth=2)
    ax.fill(angles, values, color=colors[idx], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(['Orders', 'Avg Value', 'Reorder', 'Discounts', 'Sessions', 'Member'],
                       fontsize=8)
    ax.set_title(seg, pad=15, fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('radar_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Platform preference by segment
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

seg_platform = pd.crosstab(df['segment'], df['platform'], normalize='index') * 100
seg_platform.plot(kind='bar', ax=axes[0], color=['#00B4D8', '#FF6B35', '#8338EC'],
                  edgecolor='white')
axes[0].set_title('Platform Preference by Segment (%)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)
axes[0].legend(title='Platform')

# Monthly spend by segment boxplot
seg_order = list(cluster_names.values())
df_box = [df[df['segment'] == s]['monthly_spend'].values for s in seg_order]
bp = axes[1].boxplot(df_box, patch_artist=True, labels=[s.split(' ')[1] for s in seg_order])
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Monthly Spend Distribution by Segment')
axes[1].set_ylabel('₹ Monthly Spend')

plt.tight_layout()
plt.savefig('segment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 💡 Step 9: Business Insights & Recommendations

In [ ]:
insights = """
╔══════════════════════════════════════════════════════════════════════╗
║      ZEPTO & BLINKIT CHENNAI — CUSTOMER SEGMENT INSIGHTS            ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  💎 PREMIUM LOYALISTS                                                ║
║  • High AOV (₹700+), frequent orders (20+/month), low churn          ║
║  • Strategy: Exclusive membership perks, early access, premium SKUs  ║
║                                                                      ║
║  🧪 OCCASIONAL BROWSERS                                              ║
║  • Low orders (3–6/month), irregular usage, trial-phase users        ║
║  • Strategy: Re-engagement nudges, first-order deals, reminders      ║
║                                                                      ║
║  🔥 BARGAIN HUNTERS                                                  ║
║  • High discount usage (60%+), moderate orders, price-sensitive      ║
║  • Strategy: Flash sale targeting, bundle deals, coupon push         ║
║                                                                      ║
║  ⭐ REGULAR MAINSTREAM                                               ║
║  • Steady mid-range orders, balanced spend, good retention potential ║
║  • Strategy: Loyalty rewards, subscription push, category upsell     ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(insights)

## 💾 Step 10: Save Model Artifacts

In [ ]:
import json

# Save model, scaler, PCA
joblib.dump(kmeans, 'kmeans_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(pca, 'pca_model.pkl')

# Save dataset
df.to_csv('chennai_customer_segments.csv', index=False)

# Save cluster metadata
cluster_meta = {
    'cluster_names': cluster_names,
    'features': features,
    'optimal_k': OPTIMAL_K,
    'silhouette_score': round(final_silhouette, 4)
}
with open('cluster_metadata.json', 'w') as f:
    json.dump(cluster_meta, f, indent=2)

print('✅ All artifacts saved!')
print('  → kmeans_model.pkl')
print('  → scaler.pkl')
print('  → pca_model.pkl')
print('  → chennai_customer_segments.csv')
print('  → cluster_metadata.json')

In [ ]:
# Download all files from Colab
from google.colab import files

for fname in ['kmeans_model.pkl', 'scaler.pkl', 'pca_model.pkl',
              'chennai_customer_segments.csv', 'cluster_metadata.json']:
    files.download(fname)

print('📥 Downloads triggered! Place all files in the same folder as app.py')

---
## ✅ Next Step
Download the `.pkl`, `.csv`, and `.json` files above, place them alongside `app.py`, then run:
```bash
pip install streamlit pandas scikit-learn plotly joblib
streamlit run app.py
```